# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports a checkpoint to `/kaggle/working`.

Inputs: competition data (auto-mounted), `knee-mined-labels` private dataset.
Requires the `WANDB_API_KEY` Kaggle secret.

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
COMMIT = "main"  # TODO: pin to a SHA per run
%pip install -q "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee[train]"

from knee.paths import paths  # resolves /kaggle/input + /kaggle/working automatically

# TODO(issue #4): modules below don't exist yet
# from knee.train import TrainConfig, run_training

In [ ]:
# Mined labels + split are Competition Data derivatives -> arrive via a PRIVATE
# Kaggle dataset (never the public repo).
import pandas as pd

labels = pd.read_csv("/kaggle/input/knee-mined-labels/mined_labels.csv")
split = pd.read_csv("/kaggle/input/knee-mined-labels/train_val_split.csv")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
run = wandb.init(project="rsna-knee", config={"commit": COMMIT})

In [ ]:
# TODO(issue #4): real TrainConfig once knee.train exists. Sketch:
# cfg = TrainConfig(
#     backbone="resnet34",
#     series_types=["FLUID_SAG"],   # baseline: fluid-sensitive sagittal + fallback
#     epochs=10,
#     batch_size=16,
#     limit_studies=None,           # e.g. 300 for a quota-cheap smoke run
# )

In [ ]:
# TODO(issue #4): the actual run. Competition DICOMs are pre-mounted read-only.
# result = run_training(cfg, paths, labels, split, log=wandb.log)
# print(result.val_macro_auc, result.per_label_auc)

In [ ]:
# /kaggle/working persists as notebook output; publish it as the knee-weights
# dataset afterwards so the inference notebook can attach it.
# result.save_checkpoint("/kaggle/working/baseline_v1.pt")
run.finish()